In [1]:
# ============================================================
# EXPERIMENT 007
# XGBoost with behavioural feature engineering
# ============================================================

from pathlib import Path
from time import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

EXPERIMENT_ID = "EXP-007"

RANDOM_STATE = 42
N_SPLITS = 3

TARGET = "addicted_label"
ID_COLUMN = "id"

EXP_005_OOF_AUC = 0.963659

MINIMUM_IMPROVEMENT = 0.0002
SUBMISSION_THRESHOLD = (
    EXP_005_OOF_AUC + MINIMUM_IMPROVEMENT
)


PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent


DATA_DIR = PROJECT_DIR / "data"
SUBMISSION_DIR = PROJECT_DIR / "submissions"

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = (
    DATA_DIR / "sample_submission.csv"
)


# ============================================================
# 2. LOAD DATA
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH
)


print(f"Train shape:             {train.shape}")
print(f"Test shape:              {test.shape}")
print(
    f"Sample submission shape: "
    f"{sample_submission.shape}"
)


assert TARGET in train.columns
assert TARGET not in test.columns
assert len(test) == len(sample_submission)


# ============================================================
# 3. CREATE ORIGINAL FEATURES AND TARGET
# ============================================================

X = train.drop(
    columns=[TARGET, ID_COLUMN]
).copy()

y = train[TARGET].astype(int).copy()

X_test = test.drop(
    columns=[ID_COLUMN]
).copy()


assert list(X.columns) == list(X_test.columns)

original_features = X.columns.tolist()


# ============================================================
# 4. ADD ORIGINAL MISSINGNESS INDICATORS
# ============================================================

columns_with_missing_values = [
    column
    for column in original_features
    if (
        X[column].isna().any()
        or X_test[column].isna().any()
    )
]


for column in columns_with_missing_values:

    indicator_name = f"{column}__missing"

    X[indicator_name] = (
        X[column]
        .isna()
        .astype("int8")
    )

    X_test[indicator_name] = (
        X_test[column]
        .isna()
        .astype("int8")
    )


missing_indicator_columns = [
    f"{column}__missing"
    for column in columns_with_missing_values
]


# ============================================================
# 5. FEATURE-ENGINEERING FUNCTIONS
# ============================================================

def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series
) -> pd.Series:
    """
    Divide two Series while converting division by zero
    and infinite results into missing values.
    """

    safe_denominator = denominator.replace(
        0,
        np.nan
    )

    result = numerator / safe_denominator

    return result.replace(
        [np.inf, -np.inf],
        np.nan
    )


def add_behavioral_features(
    dataframe: pd.DataFrame
) -> pd.DataFrame:
    """
    Add interpretable behavioural interaction and ratio features.
    """

    dataframe = dataframe.copy()

    # Combined leisure-oriented usage.
    dataframe["leisure_screen_time"] = (
        dataframe["social_media_hours"]
        + dataframe["gaming_hours"]
    )

    # How much of reported daily screen time is explained
    # by social media and gaming.
    dataframe["leisure_share_of_screen"] = safe_divide(
        dataframe["leisure_screen_time"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["social_media_share_of_screen"] = safe_divide(
        dataframe["social_media_hours"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["gaming_share_of_screen"] = safe_divide(
        dataframe["gaming_hours"],
        dataframe["daily_screen_time_hours"]
    )

    # Difference between weekend and typical daily usage.
    dataframe["weekend_screen_change"] = (
        dataframe["weekend_screen_time"]
        - dataframe["daily_screen_time_hours"]
    )

    dataframe["weekend_to_daily_screen_ratio"] = safe_divide(
        dataframe["weekend_screen_time"],
        dataframe["daily_screen_time_hours"]
    )

    # Screen use compared with recovery and productive activity.
    dataframe["screen_sleep_gap"] = (
        dataframe["daily_screen_time_hours"]
        - dataframe["sleep_hours"]
    )

    dataframe["screen_to_sleep_ratio"] = safe_divide(
        dataframe["daily_screen_time_hours"],
        dataframe["sleep_hours"]
    )

    dataframe["screen_to_work_ratio"] = safe_divide(
        dataframe["daily_screen_time_hours"],
        dataframe["work_study_hours"]
    )

    dataframe["leisure_to_work_ratio"] = safe_divide(
        dataframe["leisure_screen_time"],
        dataframe["work_study_hours"]
    )

    # Digital interruption and checking intensity.
    dataframe["notifications_per_screen_hour"] = safe_divide(
        dataframe["notifications_per_day"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["app_opens_per_screen_hour"] = safe_divide(
        dataframe["app_opens_per_day"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["notifications_per_app_open"] = safe_divide(
        dataframe["notifications_per_day"],
        dataframe["app_opens_per_day"]
    )

    return dataframe


# ============================================================
# 6. ADD BEHAVIOURAL FEATURES
# ============================================================

features_before_engineering = set(X.columns)

X = add_behavioral_features(X)
X_test = add_behavioral_features(X_test)

assert list(X.columns) == list(X_test.columns)


engineered_features = [
    column
    for column in X.columns
    if column not in features_before_engineering
]


print(f"\nOriginal features:        {len(original_features)}")
print(
    f"Missing indicators:      "
    f"{len(missing_indicator_columns)}"
)
print(
    f"Engineered features:     "
    f"{len(engineered_features)}"
)
print(f"Total model features:     {X.shape[1]}")

print("\nEngineered features:")

for feature in engineered_features:
    print(f"- {feature}")


print("\nEngineered feature summary:")

print(
    X[engineered_features]
    .describe()
    .T[
        [
            "count",
            "mean",
            "std",
            "min",
            "max"
        ]
    ]
    .round(4)
)


# ============================================================
# 7. IDENTIFY FEATURE TYPES
# ============================================================

categorical_columns = X.select_dtypes(
    include=[
        "object",
        "category",
        "bool"
    ]
).columns.tolist()


numeric_columns = X.columns.difference(
    categorical_columns
).tolist()


print(f"\nNumerical features:   {len(numeric_columns)}")
print(
    f"Categorical features: "
    f"{len(categorical_columns)}"
)

print("\nCategorical columns:")
print(categorical_columns)


# ============================================================
# 8. PREPROCESSING
# ============================================================

try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse_output=True
    )

except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse=True
    )


numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "one_hot",
            one_hot_encoder
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ],
    remainder="drop"
)


# ============================================================
# 9. XGBOOST CONFIGURATION
# ============================================================

# Identical to EXP-005.

model_parameters = {
    "objective": "binary:logistic",
    "eval_metric": "auc",

    "n_estimators": 3000,
    "learning_rate": 0.05,

    "max_depth": 6,
    "min_child_weight": 5,

    "subsample": 0.80,
    "colsample_bytree": 0.80,

    "reg_alpha": 0.0,
    "reg_lambda": 1.0,

    "tree_method": "hist",

    "early_stopping_rounds": 100,

    "random_state": RANDOM_STATE,
    "n_jobs": -1
}


# ============================================================
# 10. STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


out_of_fold_predictions = np.zeros(
    len(train),
    dtype=float
)

test_predictions = np.zeros(
    len(test),
    dtype=float
)


fold_scores = []
best_iterations = []
fold_feature_importances = []

experiment_start = time()


for fold_number, (
    train_indices,
    validation_indices
) in enumerate(
    cv.split(X, y),
    start=1
):

    fold_start = time()

    X_train_fold = X.iloc[
        train_indices
    ]

    X_validation_fold = X.iloc[
        validation_indices
    ]

    y_train_fold = y.iloc[
        train_indices
    ]

    y_validation_fold = y.iloc[
        validation_indices
    ]


    fold_preprocessor = clone(
        preprocessor
    )


    X_train_processed = (
        fold_preprocessor.fit_transform(
            X_train_fold
        )
    )

    X_validation_processed = (
        fold_preprocessor.transform(
            X_validation_fold
        )
    )

    X_test_processed = (
        fold_preprocessor.transform(
            X_test
        )
    )


    if fold_number == 1:
        print(
            "\nProcessed training matrix shape:",
            X_train_processed.shape
        )


    model = XGBClassifier(
        **model_parameters
    )


    model.fit(
        X_train_processed,
        y_train_fold,
        eval_set=[
            (
                X_validation_processed,
                y_validation_fold
            )
        ],
        verbose=False
    )


    validation_probabilities = (
        model.predict_proba(
            X_validation_processed
        )[:, 1]
    )

    fold_test_probabilities = (
        model.predict_proba(
            X_test_processed
        )[:, 1]
    )


    out_of_fold_predictions[
        validation_indices
    ] = validation_probabilities


    test_predictions += (
        fold_test_probabilities
        / N_SPLITS
    )


    fold_auc = roc_auc_score(
        y_validation_fold,
        validation_probabilities
    )


    fold_scores.append(
        float(fold_auc)
    )


    best_iteration = getattr(
        model,
        "best_iteration",
        None
    )

    best_iterations.append(
        best_iteration
    )


    transformed_feature_names = (
        fold_preprocessor
        .get_feature_names_out()
    )


    fold_importance = pd.Series(
        model.feature_importances_,
        index=transformed_feature_names,
        name=f"fold_{fold_number}"
    )


    fold_feature_importances.append(
        fold_importance
    )


    fold_minutes = (
        time() - fold_start
    ) / 60


    print(
        f"Fold {fold_number}/{N_SPLITS} | "
        f"AUC: {fold_auc:.6f} | "
        f"Best iteration: {best_iteration} | "
        f"Time: {fold_minutes:.2f} minutes"
    )


# ============================================================
# 11. VALIDATION RESULTS
# ============================================================

overall_oof_auc = roc_auc_score(
    y,
    out_of_fold_predictions
)


mean_fold_auc = float(
    np.mean(fold_scores)
)

std_fold_auc = float(
    np.std(fold_scores)
)


difference_vs_exp_005 = (
    overall_oof_auc
    - EXP_005_OOF_AUC
)


total_minutes = (
    time() - experiment_start
) / 60


print("\n" + "=" * 60)
print("EXPERIMENT 007 RESULTS")
print("=" * 60)


for fold_number, score in enumerate(
    fold_scores,
    start=1
):
    print(
        f"Fold {fold_number} AUC: "
        f"{score:.6f}"
    )


print(
    f"\nMean fold AUC:          "
    f"{mean_fold_auc:.6f}"
)

print(
    f"Fold AUC SD:            "
    f"{std_fold_auc:.6f}"
)

print(
    f"OOF AUC:                "
    f"{overall_oof_auc:.6f}"
)

print(
    f"EXP-005 OOF:            "
    f"{EXP_005_OOF_AUC:.6f}"
)

print(
    f"Difference vs EXP-005:  "
    f"{difference_vs_exp_005:+.6f}"
)

print(
    f"Best iterations:        "
    f"{best_iterations}"
)

print(
    f"Runtime:                "
    f"{total_minutes:.2f} minutes"
)


# ============================================================
# 12. INTERPRET RESULT
# ============================================================

print("\nInterpretation:")


if (
    difference_vs_exp_005
    >= MINIMUM_IMPROVEMENT
):
    print(
        "Behavioural feature engineering produced a useful "
        "improvement. Ratios and imbalances contain signal "
        "beyond the original raw variables."
    )

elif (
    difference_vs_exp_005
    > -MINIMUM_IMPROVEMENT
):
    print(
        "The result is effectively unchanged. XGBoost was "
        "already learning most of these relationships from "
        "the original features."
    )

else:
    print(
        "The engineered features reduced validation performance. "
        "They likely added redundant or noisy representations."
    )


# ============================================================
# 13. FEATURE IMPORTANCE
# ============================================================

feature_importance_table = pd.concat(
    fold_feature_importances,
    axis=1
).fillna(0)


feature_importance_table["mean_importance"] = (
    feature_importance_table.mean(axis=1)
)


feature_importance_table["importance_sd"] = (
    feature_importance_table[
        [
            "fold_1",
            "fold_2",
            "fold_3"
        ]
    ].std(axis=1)
)


feature_importance_table = (
    feature_importance_table
    .sort_values(
        "mean_importance",
        ascending=False
    )
)


print("\nTop 30 transformed features:")

print(
    feature_importance_table[
        [
            "mean_importance",
            "importance_sd"
        ]
    ]
    .head(30)
    .round(6)
)


engineered_importance_mask = (
    feature_importance_table.index
    .to_series()
    .apply(
        lambda name: any(
            feature in name
            for feature in engineered_features
        )
    )
)


engineered_feature_importance = (
    feature_importance_table.loc[
        engineered_importance_mask,
        [
            "mean_importance",
            "importance_sd"
        ]
    ]
    .sort_values(
        "mean_importance",
        ascending=False
    )
)


print("\nEngineered feature importance:")

print(
    engineered_feature_importance
    .round(6)
)


# ============================================================
# 14. PREDICTION SANITY CHECKS
# ============================================================

prediction_summary = pd.Series(
    test_predictions,
    name="predicted_probability"
).describe()


print("\nTest prediction summary:")
print(prediction_summary)

print(
    f"\nExact minimum: "
    f"{test_predictions.min():.12f}"
)

print(
    f"Exact maximum: "
    f"{test_predictions.max():.12f}"
)


assert np.isfinite(
    out_of_fold_predictions
).all()

assert np.isfinite(
    test_predictions
).all()


FLOAT_TOLERANCE = 1e-6


assert test_predictions.min() >= -FLOAT_TOLERANCE

assert test_predictions.max() <= (
    1 + FLOAT_TOLERANCE
)


test_predictions = np.clip(
    test_predictions,
    0.0,
    1.0
)


assert (
    (test_predictions >= 0)
    & (test_predictions <= 1)
).all()

assert np.std(test_predictions) > 0


# ============================================================
# 15. CONDITIONAL SUBMISSION
# ============================================================

if overall_oof_auc >= SUBMISSION_THRESHOLD:

    submission = sample_submission.copy()

    assert TARGET in submission.columns

    submission[TARGET] = test_predictions


    submission_path = (
        SUBMISSION_DIR
        / "exp_007_behavioral_features.csv"
    )


    submission.to_csv(
        submission_path,
        index=False
    )


    print(
        "\nSubmission created because EXP-007 improved "
        "upon EXP-005 by at least "
        f"{MINIMUM_IMPROVEMENT:.4f}."
    )

    print(f"Saved to:\n{submission_path}")

    print("\nSubmission preview:")
    print(submission.head())


else:
    submission_path = None

    print(
        "\nNo submission created because OOF AUC did not reach "
        f"{SUBMISSION_THRESHOLD:.6f}."
    )

Train shape:             (691369, 14)
Test shape:              (296302, 13)
Sample submission shape: (296302, 2)

Original features:        12
Missing indicators:      12
Engineered features:     13
Total model features:     37

Engineered features:
- leisure_screen_time
- leisure_share_of_screen
- social_media_share_of_screen
- gaming_share_of_screen
- weekend_screen_change
- weekend_to_daily_screen_ratio
- screen_sleep_gap
- screen_to_sleep_ratio
- screen_to_work_ratio
- leisure_to_work_ratio
- notifications_per_screen_hour
- app_opens_per_screen_hour
- notifications_per_app_open

Engineered feature summary:
                                  count     mean      std     min       max
leisure_screen_time            484281.0   3.9303   1.6793  0.0100   11.2900
leisure_share_of_screen        451246.0   0.5241   0.1478  0.0039    0.9470
social_media_share_of_screen   502260.0   0.3269   0.1344  0.0000    0.8853
gaming_share_of_screen         507723.0   0.1971   0.1111  0.0000    0.7995
we

### Experiment Log

In [2]:
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd


EXPERIMENT_LOG_PATH = PROJECT_DIR / "experiment_log.csv"


def log_experiment(
    experiment_id,
    description,
    model,
    features,
    validation_method,
    cv_scores,
    kaggle_score=None,
    changes="",
    submission_file="",
    notes="",
    log_path=EXPERIMENT_LOG_PATH
):
    """
    Add or update one experiment in experiment_log.csv.

    If the experiment_id already exists, its previous row is replaced.
    """

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    cv_scores = [float(score) for score in cv_scores]

    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    kaggle_score_value = (
        float(kaggle_score)
        if kaggle_score is not None
        else np.nan
    )

    kaggle_cv_gap = (
        kaggle_score_value - cv_mean
        if pd.notna(kaggle_score_value)
        else np.nan
    )

    experiment_record = {
        "experiment_id": experiment_id,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "description": description,
        "model": model,
        "features": json.dumps(list(features)),
        "n_features": len(features),
        "validation_method": validation_method,
        "cv_scores": json.dumps(cv_scores),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "kaggle_score": kaggle_score_value,
        "kaggle_cv_gap": kaggle_cv_gap,
        "changes": changes,
        "submission_file": submission_file,
        "notes": notes
    }

    if log_path.exists():
        experiments = pd.read_csv(log_path)

        # Prevent duplicate rows when rerunning the same experiment cell.
        if "experiment_id" in experiments.columns:
            experiments = experiments[
                experiments["experiment_id"] != experiment_id
            ].copy()
    else:
        experiments = pd.DataFrame()

    new_row = pd.DataFrame([experiment_record])

    experiments = pd.concat(
        [experiments, new_row],
        ignore_index=True
    )

    experiments = experiments.sort_values(
        by="experiment_id"
    ).reset_index(drop=True)

    experiments.to_csv(log_path, index=False)

    print(f"Logged {experiment_id}")
    print(f"CV mean:       {cv_mean:.6f}")
    print(f"CV SD:         {cv_std:.6f}")

    if pd.notna(kaggle_score_value):
        print(f"Kaggle score:  {kaggle_score_value:.6f}")
        print(f"Kaggle-CV gap: {kaggle_cv_gap:+.6f}")

    print(f"Log saved to:  {log_path}")

    return experiments

NameError: name 'PROJECT_DIR' is not defined

In [ ]:
experiments = log_experiment(
    experiment_id="EXP-007",
    description=(
        "XGBoost model using original features, missingness indicators, "
        "and engineered behavioural ratios and interaction features."
    ),
    model="XGBClassifier",
    features=X.columns.tolist(),
    validation_method="3-fold StratifiedKFold with ROC AUC",
    cv_scores=[
        0.963396,
        0.964245,
        0.964271
    ],
    kaggle_score=0.96559,
    changes=(
        "Retained the EXP-005 model configuration and added engineered "
        "features representing leisure screen time, usage shares, weekend "
        "usage changes, screen-to-sleep and screen-to-work imbalances, "
        "and notification and app-opening intensity."
    ),
    submission_file="exp_007_behavioral_features.csv",
    notes=(
        "OOF AUC was 0.963969 with fold SD 0.000407, improving upon "
        "EXP-005 by 0.000310. Kaggle improved from 0.965340 to 0.965590, "
        "a gain of 0.000250. Best iterations were 2369, 2312, and 2194. "
        "The close agreement between local and leaderboard gains supports "
        "the hypothesis that behavioural ratios and imbalances add real signal."
    )
)

experiments.tail()